# Fiscal policy in RANK and TANK

The model is the IKC model of the lecture: real rate rule ($r_t=r_{ss}$), real bonds and no markups ($\mu=1$).
The government follows the debt rule $dB_t=\rho_B\left(dB_{t-1}+dG_t\right)$ and taxes adjust, $T_t=\tau_tY_t$.
A share $\lambda$ of households is hand-to-mouth; `par.lam = 0` is RANK.
Unions set wages through the W-NKPC, and the central bank sets $1+i_t=(1+r_{ss})(1+\pi_{t+1})$.
The calibration is annual: $r_{ss}=0.02$, $G_{ss}=0.2$, $B_{ss}=0.25$.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import RANKModel

## Settings

In [ ]:
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black', 'grid.alpha': 0.25, 'grid.linestyle': '--', 'font.size': 12})

## Steady state

In [ ]:
model = RANKModel.RANKModelClass(name='RANK')
model.find_ss(do_print=True)
model.test_path()

## Q1

Read the model equations in `blocks.py`. Check that they match the IKC model in RANK and TANK from the slides.

*Hint:* list the unknowns and targets in `RANKModel.py`, then write each block of `blocks.py` as an equation and compare with the "Simplified system of equations" slide.

## Q2

Solve for the IRF to a government spending shock in the RANK model using `model.find_IRFs()`. Compare the solution to the analytical formula derived in the slides.

*Hint:* in RANK, $\boldsymbol{M}^{RA}=(1-\beta)\boldsymbol{1}\boldsymbol{q}^{\prime}$ with $q_t=(1+r_{ss})^{-t}$, which you can build with `np.outer(np.ones(par.T),q)`. What is $\boldsymbol{M}^{RA}(d\boldsymbol{G}-d\boldsymbol{T})$? You can also solve the truncated IKC $(\boldsymbol{I}-\boldsymbol{M})d\boldsymbol{Y}=d\boldsymbol{G}-\boldsymbol{M}d\boldsymbol{T}$ with `np.linalg.solve(A,b)`.

In [ ]:
model.compute_jacs()
model.find_IRFs(shocks=['G'])

par = model.par
ss = model.ss
dY = model.IRF['Y']
dG = model.IRF['G']
dT = model.IRF['taxes']
q = (1+ss.r)**(-np.arange(par.T))

# a. analytical solution from the slides
dY_analytical = ...

# b. IKC with the RANK iMPC matrix
M_RA = ...
dY_IKC = ...

print(f'present value of the deficit: {q@(dG-dT):.1e}')
print(f'max |dY - analytical|: {np.max(np.abs(dY-dY_analytical)):.1e}')
print(f'max |dY - dY_IKC|:     {np.max(np.abs(dY-dY_IKC)):.1e}')

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(dY[:30],label='find_IRFs')
axes[0].plot(dY_analytical[:30],ls='--',label='analytical')
axes[0].set_title('output, $dY$')
axes[0].legend()
axes[1].plot(dG[:30],label='$dG$')
axes[1].plot(dT[:30],label='$dT$')
axes[1].plot(model.IRF['B'][:30],label='$dB$')
axes[1].set_title('government')
axes[1].legend()
for ax in axes: ax.set_xlabel('years')
fig.tight_layout()

## Q3

Compute the impact and cumulative multipliers in RANK, using your previous answer. What feature of the household block explains this result?

*Hint:* the impact multiplier is $dY_0/dG_0$ and the cumulative multiplier is $\boldsymbol{q}^{\prime}d\boldsymbol{Y}/\boldsymbol{q}^{\prime}d\boldsymbol{G}$ (use `q@dY`).

In [ ]:
impact = ...
cumulative = ...
print(f'RANK: impact = {impact:.3f}, cumulative = {cumulative:.3f}')

## Q4

Solve for the IRF to a government spending shock in the TANK model (`par.lam = 0.3`) using `model.find_IRFs()`. Compare the solution to the analytical formula derived in the slides.

*Hint:* make a copy with `model.copy()`, change `par.lam`, then solve the steady state, the Jacobians and the IRFs again. The analytical solution is on the "Solving TANK" slide.

In [ ]:
lam = 0.3

model_TANK = model.copy(name='TANK')
model_TANK.par.lam = lam

# a. solve the steady state, the Jacobians and the IRFs
...

# b. compare with the analytical solution
dY_TANK = model_TANK.IRF['Y']
dT_TANK = model_TANK.IRF['taxes']
dY_analytical = ...
print(f'max |dY - analytical|: {np.max(np.abs(dY_TANK-dY_analytical)):.1e}')

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(dY_TANK[:30],label='find_IRFs')
ax.plot(dY_analytical[:30],ls='--',label='analytical')
ax.plot(dG[:30],ls=':',color='black',label='$dG$')
ax.set_title('output in TANK, $dY$')
ax.set_xlabel('years')
ax.legend();

## Q5

Compute the impact and cumulative multipliers in TANK. What feature of the household block explains this result?

*Hint:* what is $dT_0$ under the debt rule? Compare the impact multiplier with $1+\frac{\lambda}{1-\lambda}\rho_B$.

In [ ]:
impact_TANK = ...
cumulative_TANK = ...
print(f'TANK: impact = {impact_TANK:.3f}, cumulative = {cumulative_TANK:.3f}')

## Q6

Compute the impact and cumulative multipliers for $\lambda \in \{0, 0.1, 0.2, 0.3, 0.4, 0.5\}$, and discuss your results.

*Hint:* reuse the steps of Q4 inside a loop.

In [ ]:
lams = [0.0,0.1,0.2,0.3,0.4,0.5]
impacts = []
cumulatives = []

for lam_ in lams:

    model_ = model.copy(name=f'lam={lam_}')

    # a. set lambda and solve the model
    ...

    # b. store the multipliers
    impacts.append(...)
    cumulatives.append(...)
    print(f'lam = {lam_:.1f}: impact = {impacts[-1]:.3f}, cumulative = {cumulatives[-1]:.3f}')

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(lams,impacts,marker='o',label='impact')
ax.plot(lams,cumulatives,marker='o',label='cumulative')
ax.set_xlabel('$\\lambda$')
ax.set_title('fiscal multipliers in TANK')
ax.legend();

## Q7

Compute the inflation response to the shock in RANK and TANK. Does inflation have any impact on the output response to the shock? Why?

*Hint:* inflation is `model.IRF['pi']`. To see whether inflation matters for output, change the slope of the W-NKPC, `par.kappa`, and check whether $d\boldsymbol{Y}$ moves.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(model.IRF['pi'][:30],label='RANK')
axes[0].plot(model_TANK.IRF['pi'][:30],label='TANK')
axes[0].set_title('inflation, $d\\pi$')
axes[1].plot(model.IRF['Y'][:30],label='RANK')
axes[1].plot(model_TANK.IRF['Y'][:30],label='TANK')
axes[1].set_title('output, $dY$')

# same models with more flexible wages
for model_, label in [(model,'RANK'),(model_TANK,'TANK')]:

    model_flex = model_.copy(name=f'{label}, flexible wages')

    # a. raise kappa and solve the IRFs again
    ...

    axes[0].plot(model_flex.IRF['pi'][:30],ls='--',label=f'{label}, 4x kappa')
    axes[1].plot(model_flex.IRF['Y'][:30],ls='--',label=f'{label}, 4x kappa')

for ax in axes:
    ax.set_xlabel('years')
    ax.legend()
fig.tight_layout()